<a href="https://colab.research.google.com/github/sanjay0423/AgenticAI/blob/main/Sanjay_LangSmith_Demo_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Agent Evaluation 1 - Observability**

This module demonstrates using LangSmith to monitor your agent in production

Observability is important for any software application, but especially so for LLM applications. LLMs are non-deterministic by nature, meaning they can produce unexpected results. This makes them trickier than normal to debug.

Note that observability is important throughout all stages of application development - from prototyping, to beta testing, to production. There are different considerations at all stages, but they are all intricately tied together. In this tutorial we walk through the natural progression.

Let's assume that we're building a simple RAG application using the OpenAI SDK. The simple application we're adding observability to looks like.

### Colab secrets (required)

1. Open **Runtime → Secrets** (lock icon in the left sidebar).
2. Add **`OPENAI_API_KEY`** → your OpenAI API key. Turn **Notebook access** ON.
3. Add **`LANGSMITH_API_KEY`** → from [smith.langchain.com](https://smith.langchain.com) → **Settings** → **API Keys**. Turn **Notebook access** ON.

**Do not** paste keys into notebook cells — GitHub *push protection* will reject commits that contain secrets.

The next cell loads both into `os.environ` via `userdata.get(...)` only (safe for git).

---
Open traces in **LangSmith**: [smith.langchain.com](https://smith.langchain.com)

In [12]:
# Install dependencies (Colab)
%pip install -q langsmith openai

import os
from openai import OpenAI
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from google.colab import userdata

# LangSmith / LangChain tracing env (no secrets here)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = os.environ.get("LANGSMITH_PROJECT", "default")

# Load API keys from Colab Secrets (names must match exactly)
_openai_key = userdata.get("OPENAI_API_KEY")
_langsmith_key = userdata.get("LANGSMITH_API_KEY")

if not _openai_key or not _langsmith_key:
    raise RuntimeError(
        "Missing Colab secrets. Add OPENAI_API_KEY and LANGSMITH_API_KEY under Runtime → Secrets "
        "and enable notebook access for both."
    )

os.environ["OPENAI_API_KEY"] = _openai_key
os.environ["LANGSMITH_API_KEY"] = _langsmith_key

print("OK: Keys loaded from userdata into os.environ (not stored in the notebook file).")


OK: Keys loaded from userdata into os.environ (not stored in the notebook file).


# **Trace your LLM calls**

The first thing you might want to trace is all your OpenAI calls. After all, this is where the LLM is actually being called, so it is the most important part! We've tried to make this as easy as possible with LangSmith by introducing a dead-simple OpenAI wrapper. All you have to do is modify your code to look something like:

In [13]:
#start
openai_client = wrap_openai(OpenAI())

def retriever(query: str):
    results = ["Harrison worked at Kensho"]
    return results

def rag(question):
    docs = retriever(question)
    system_message = """Answer the users question using only the provided information below:

    {docs}""".format(docs="\n".join(docs))

    response = openai_client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": question},
        ],
        model="gpt-4o-mini",
    )
    return response.choices[0].message.content

Notice how we import from langsmith.wrappers import wrap_openai and use it to wrap the OpenAI client (openai_client = wrap_openai(OpenAI())).

What happens if you call it in the following way?

In [14]:
rag("where did harrison work")

'Harrison worked at Kensho.'

In [15]:
openai_client = wrap_openai(OpenAI())


def retriever(query: str):
    results = ["Harrison worked at Kensho"]
    return results

@traceable
def rag(question):
    docs = retriever(question)
    system_message = """Answer the users question using only the provided information below:

    {docs}""".format(docs="\n".join(docs))

    response = openai_client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": question},
        ],
        model="gpt-4o-mini",
    )
    return response.choices[0].message.content

Notice how we import from langsmith import traceable and use it decorate the overall function (@traceable).

What happens if you call it in the following way?

In [16]:
rag("where did harrison work")

'Harrison worked at Kensho.'

# **Trace the retrieval step**
There's one last part of the application we haven't traced - the retrieval step! Retrieval is a key part of LLM applications, and we've made it easy to log retrieval steps as well. All we have to do is modify our code to look like:

In [17]:
openai_client = wrap_openai(OpenAI())

@traceable(run_type="retriever")
def retriever(query: str):
    results = ["Harrison worked at Kensho"]
    return results

@traceable
def rag(question):
    docs = retriever(question)
    system_message = """Answer the users question using only the provided information below:

    {docs}""".format(docs="\n".join(docs))

    response = openai_client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": question},
        ],
        model="gpt-4o-mini",
    )
    return response.choices[0].message.content

Notice how we import from langsmith import traceable and use it decorate the overall function (@traceable(run_type="retriever")).

What happens if you call it in the following way?

In [18]:
rag("where did harrison work")

'Harrison worked at Kensho.'

# **Beta Testing**
The next stage of LLM application development is beta testing your application. This is when you release it to a few initial users. Having good observability set up here is crucial as often you don't know exactly how users will actually use your application, so this allows you get insights into how they do so. This also means that you probably want to make some changes to your tracing set up to better allow for that. This extends the observability you set up in the previous section

# **Collecting Feedback**
A huge part of having good observability during beta testing is collecting feedback. What feedback you collect is often application specific - but at the very least a simple thumbs up/down is a good start. After logging that feedback, you need to be able to easily associate it with the run that caused that. Luckily LangSmith makes it easy to do that.

First, you need to log the feedback from your app. An easy way to do this is to keep track of a run ID for each run, and then use that to log feedback. Keeping track of the run ID would look something like:

In [19]:
import uuid

run_id = str(uuid.uuid4())
print(run_id)


rag(
    "where did harrison work",
    langsmith_extra={"run_id": run_id, "metadata": {"user_id": "harrison"}}
)

from langsmith import Client
ls_client = Client()

ls_client.create_feedback(
    run_id,
    key="user-score",
    score=1.0,
)


eb825057-de08-4d37-a3a3-7daad5df8ea5


Feedback(id=UUID('019d1627-181a-7441-951b-cfbeb21f16eb'), created_at=datetime.datetime(2026, 3, 22, 15, 25, 54, 74107, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 3, 22, 15, 25, 54, 74112, tzinfo=datetime.timezone.utc), run_id=UUID('eb825057-de08-4d37-a3a3-7daad5df8ea5'), trace_id=None, key='user-score', score=1.0, value=None, comment=None, correction=None, feedback_source=FeedbackSourceBase(type='api', metadata={}, user_id=None, user_name=None), session_id=None, start_time=None, comparative_experiment_id=None, feedback_group_id=None, extra=None)

In [20]:
import uuid

run_id = str(uuid.uuid4())
print(run_id)


rag(
    "where did peter work",
    langsmith_extra={"run_id": run_id, "metadata": {"user_id": "peter"}}
)

from langsmith import Client
ls_client = Client()

ls_client.create_feedback(
    run_id,
    key="user-score",
    score=0.0,
)


1c6397b1-dcb1-428c-ab7f-454d676b1d2d


Feedback(id=UUID('019d1627-2130-7f71-a2bb-9d22e67461ab'), created_at=datetime.datetime(2026, 3, 22, 15, 25, 56, 400807, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 3, 22, 15, 25, 56, 400813, tzinfo=datetime.timezone.utc), run_id=UUID('1c6397b1-dcb1-428c-ab7f-454d676b1d2d'), trace_id=None, key='user-score', score=0.0, value=None, comment=None, correction=None, feedback_source=FeedbackSourceBase(type='api', metadata={}, user_id=None, user_name=None), session_id=None, start_time=None, comparative_experiment_id=None, feedback_group_id=None, extra=None)

# **Logging Metadata**
It is also a good idea to start logging metadata. This allows you to start keep track of different attributes of your app. This is important in allowing you to know what version or variant of your app was used to produce a given result.

For this example, we will log the LLM used. Oftentimes you may be experimenting with different LLMs, so having that information as metadata can be useful for filtering. In order to do that, we can add it as such:

In [23]:
openai_client = wrap_openai(OpenAI())

@traceable(run_type="retriever")
def retriever(query: str):
    results = ["Harrison worked at Kensho"]
    return results


@traceable(metadata={"llm": "gpt-4o-mini"})
def rag(question):
    docs = retriever(question)
    system_message = """Answer the users question using only the provided information below:

    {docs}""".format(docs="\n".join(docs))

    response = openai_client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": question},
        ],
        model="gpt-4o-mini",
    )

    return response.choices[0].message.content


In [22]:
import uuid

run_id = str(uuid.uuid4())
rag(
    "where did harrison work",
    langsmith_extra={"run_id": run_id, "metadata": {"user_id": "harrison"}}
)

'Harrison worked at Kensho.'